# ViNLI · M3 vs REL256 · BF16 · seed 42 · G4 parallel pilot

**Run All launches exactly two independent trainers on CUDA:0:**
**M3_FULL (relation width 128)** and **ARCH_REL256 (width 256)**, both seed 42.
Each run uses one CafeBERT encoder and classifier. No ensemble. Only the
relation branch width changes; all remaining settings match the current run.

Pinned CafeBERT; ViNLI max length **512**; BF16 train / FP32 evaluation;
physical batch **4 × accumulation 4 = effective 16 per process**; LR 1e-5;
weight decay .005; warmup .06; smoothing .02; dropout .1;
EMA .992 from step 100; five MSD paths; **7 epochs, patience 0**.
Dev/test scans remain every **30 optimizer steps**, with test scans after
EMA activation (first interval step 120), plus the last training step.
HF stores both dev-selected and exploratory peak-test checkpoints. The
comparison shows both endpoints separately, with accuracy and per-class F1.

Select the **96 GB G4** runtime. Default: **two concurrent processes**.
`EXECUTION` accepts 1–3 workers; this two-job pilot never launches a third
model. GPU admission checks actual free VRAM, using an **estimated** 24 GiB
per trainer and 8 GiB spare. This is not a measured peak or a memory cap.
Fewer slots are admitted when free VRAM is lower. Each child gets two CPU
threads. Increasing concurrency shares GPU compute and may not increase speed.

Add `HF_TOKEN` (model-repo write) and optional `GITHUB_TOKEN` (private
checkout, Contents: read) to Colab Secrets. Drive keeps evidence/progress;
HF keeps immutable verified weights. Source, recipe, runtime and requested
execution settings are frozen under RUN_GROUP. Changing them needs a new group.
Interrupting the training cell stops all active children. Completed verified
jobs are skipped; completed local results retry publication; interrupted
training restarts that job from scratch. Run All adds no other methods/seeds.
Test peaks are explicitly exploratory scores selected on the test set.


In [ ]:
import os
import pathlib
import subprocess
import sys
import json

REPO_URL = "https://github.com/baominh5xx2/grid-repo.git"
SOURCE_REF = "main"  # First session freezes its SHA; resume reuses that SHA.
REQUIRE_GITHUB_TOKEN = False
RUN_GROUP = "vinli-rel256-g4-bf16-pilot-2026-10-04"
DATASETS = ["vinli"]
METHODS = ["M3_FULL", "ARCH_REL256"]
SEEDS = [42]
JOB_SPECS = [dict(dataset="vinli", experiment_id=method, seed=42) for method in METHODS]
VINLI_MAX_LENGTH = 512
EXPECTED_SPLIT_SIZES = {"train": 18282, "dev": 2255, "test": 2264}
PINNED_MODEL_NAME = "uitnlp/CafeBERT"
PINNED_MODEL_REVISION = "af76fcf2a04096b2b54b348a3e4eb48253c93c5d"
PINNED_VINLI_REVISION = "47bd78ac5d075bd00a3cb4cdd3ede4eec4acf8c2"
TRAIN_PRECISION = "bf16"
EVALUATION_PRECISION = "fp32"
USE_EMA = True
USE_MSD = True
MSD_PROBABILITIES = [0.1, 0.2, 0.3, 0.4, 0.5]
WANDB_ENABLED = False
TEST_PEAK_EXPLORATORY = True
FROZEN_FINAL = False
USE_DRIVE = True
HF_PRIVATE = False
HYPERPARAMS = dict(epochs=7, eval_steps=30, patience=0, lr=1e-5,
                  weight_decay=0.005, warmup_ratio=0.06, label_smoothing=0.02,
                  dropout=0.1, physical_batch_size=4, grad_accum=4,
                  ema_decay=0.992, ema_start_step=100)
assert DATASETS == ["vinli"] and SEEDS == [42]
assert METHODS == ["M3_FULL", "ARCH_REL256"]
assert VINLI_MAX_LENGTH == 512 and USE_EMA and USE_MSD
assert not WANDB_ENABLED and TEST_PEAK_EXPLORATORY and not FROZEN_FINAL
print(f"Configured: {len(JOB_SPECS)} ViNLI seed-42 test-aware exploratory jobs; peak test selection")


EXECUTION = dict(parallel_jobs=2, per_job_vram_gib=24, reserve_vram_gib=8,
                 cpu_threads_per_job=2, poll_seconds=2)
assert 1 <= EXECUTION["parallel_jobs"] <= 3
print("Independent trainers requested:", EXECUTION["parallel_jobs"], "| jobs:", len(JOB_SPECS))


## Authenticate, freeze source and check GPU
Public checkout works without GITHUB_TOKEN. Secrets are never put in remote
URLs or manifests. Resume checks out the original frozen SHA.


In [ ]:
import contextlib
import tempfile
from getpass import getpass
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse

parsed_repo = urlparse(REPO_URL)
if (parsed_repo.scheme != "https" or parsed_repo.netloc != "github.com"
        or parsed_repo.query or parsed_repo.fragment):
    raise ValueError("Use a plain HTTPS github.com repository URL without credentials")
repo_parts = parsed_repo.path.removesuffix(".git").strip("/").split("/")
if len(repo_parts) != 2 or not all(repo_parts):
    raise ValueError("Expected https://github.com/OWNER/REPO.git")
GITHUB_REPOSITORY = "/".join(repo_parts)

def github_repo_info(token):
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "grid-colab-bootstrap"}
    if token:
        headers["Authorization"] = "Bearer " + token
    request = Request("https://api.github.com/repos/" + GITHUB_REPOSITORY, headers=headers)
    try:
        with urlopen(request, timeout=30) as response:
            info = json.load(response)
    except HTTPError as error:
        raise RuntimeError(f"GitHub access failed (HTTP {error.code}); check token, expiry and selected repository") from None
    except URLError:
        raise RuntimeError("GitHub is unreachable; retry the access cell") from None
    if info.get("full_name", "").lower() != GITHUB_REPOSITORY.lower():
        raise RuntimeError("GitHub repository identity mismatch")
    return info

@contextlib.contextmanager
def github_git_environment(token):
    env = os.environ.copy()
    env.pop("GITHUB_TOKEN", None)
    env.update(GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="3",
               GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="",
               GIT_CONFIG_KEY_1="credential.useHttpPath", GIT_CONFIG_VALUE_1="true",
               GIT_CONFIG_KEY_2="credential.username", GIT_CONFIG_VALUE_2="x-access-token")
    with tempfile.TemporaryDirectory(prefix="grid-git-auth-") as directory:
        helper = pathlib.Path(directory) / "askpass.py"
        helper.write_text(chr(10).join([
            "#!" + sys.executable, "import os, sys",
            'print("x-access-token" if "username" in sys.argv[1].lower() else os.environ.get("GRID_GITHUB_TOKEN", ""))',
            ""]), encoding="utf-8")
        helper.chmod(0o700)
        env["GIT_ASKPASS"] = str(helper)
        if token:
            env["GRID_GITHUB_TOKEN"] = token
        else:
            env.pop("GRID_GITHUB_TOKEN", None)
        yield env

def github_git(arguments, env, cwd=None):
    result = subprocess.run(["git", *arguments], cwd=cwd, env=env,
                            text=True, capture_output=True)
    if result.returncode:
        # Avoid echoing credential-bearing remote errors or subprocess arguments.
        raise RuntimeError(f"Git failed ({arguments[0]}, exit {result.returncode}); check GITHUB_TOKEN Contents: read permission and SOURCE_REF")
    return result.stdout.strip()

_github_token = os.environ.get("GITHUB_TOKEN")
if not _github_token:
    try:
        from google.colab import userdata
        _github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        _github_token = None
if not _github_token and REQUIRE_GITHUB_TOKEN:
    _github_token = getpass("GITHUB_TOKEN (Contents: read): ").strip()
    if not _github_token:
        raise RuntimeError("GITHUB_TOKEN is required by configuration")
if _github_token:
    _github_info = github_repo_info(_github_token)
else:
    try:
        _github_info = github_repo_info(None)
    except RuntimeError as error:
        if "HTTP 404" not in str(error) and "HTTP 401" not in str(error):
            raise
        _github_token = getpass("GITHUB_TOKEN for private repository (Contents: read): ").strip()
        if not _github_token:
            raise RuntimeError("Private/unavailable repository: add GITHUB_TOKEN to Colab Secrets") from None
        _github_info = github_repo_info(_github_token)
print("GitHub repository access verified:", GITHUB_REPOSITORY,
      "| authenticated" if _github_token else "| public read")


In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORKSPACE = pathlib.Path("/content") if IN_COLAB else pathlib.Path.cwd() / "bf16_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    OUTPUT_ROOT = pathlib.Path("/content/drive/MyDrive/grid-repo-runs") / RUN_GROUP
else:
    OUTPUT_ROOT = WORKSPACE / "results" / RUN_GROUP
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORKSPACE / "grid-repo"
LOCAL_RUN_ROOT = WORKSPACE / "training" / RUN_GROUP
os.environ["HF_HOME"] = str(WORKSPACE / "hf-cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
previous_manifest = OUTPUT_ROOT / "batch_manifest.json"
previous = json.loads(previous_manifest.read_text()) if previous_manifest.exists() else None
target_ref = previous["git_sha"] if previous else SOURCE_REF
with github_git_environment(_github_token) as git_env:
    if not (REPO_DIR / ".git").exists():
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            raise RuntimeError(f"Choose an empty checkout directory: {REPO_DIR}")
        github_git(["clone", "--no-checkout", REPO_URL, str(REPO_DIR)], git_env)
    else:
        if github_git(["remote", "get-url", "origin"], git_env, REPO_DIR) != REPO_URL:
            raise RuntimeError("Checkout origin differs; choose a fresh workspace directory")
        if github_git(["status", "--porcelain"], git_env, REPO_DIR):
            raise RuntimeError("Source checkout is dirty; preserve those edits before proceeding.")
    github_git(["fetch", "--depth", "1", "origin", target_ref], git_env, REPO_DIR)
    github_git(["checkout", "--detach", "FETCH_HEAD"], git_env, REPO_DIR)
    SOURCE_SHA = github_git(["rev-parse", "HEAD"], git_env, REPO_DIR)
    assert not github_git(["status", "--porcelain"], git_env, REPO_DIR)
del git_env, _github_token
print("Frozen source SHA:", SOURCE_SHA)
print("Persistent results:", OUTPUT_ROOT)


In [ ]:
if "_LOADED_SOURCE_SHA" in globals() and _LOADED_SOURCE_SHA != SOURCE_SHA:
    raise RuntimeError("Source SHA changed in this kernel. Restart the runtime kernel and rerun the setup cells.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR) + "[hf]"], check=True)
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
_LOADED_SOURCE_SHA = SOURCE_SHA


In [ ]:
import torch
from gated_dual_ema_msd.training.precision import bf16_enabled
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before starting this notebook.")
assert bf16_enabled(torch.device("cuda:0"))
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| Train: BF16 | Eval: FP32")


from gated_dual_ema_msd.operations.parallel_notebook_batch import gpu_slot_plan
GPU_PLAN = gpu_slot_plan(*torch.cuda.mem_get_info(0), requested=EXECUTION["parallel_jobs"],
                        job_count=len(JOB_SPECS), per_job_gib=EXECUTION["per_job_vram_gib"],
                        reserve_gib=EXECUTION["reserve_vram_gib"])
print("GPU scheduling estimate:", GPU_PLAN)


In [ ]:
from getpass import getpass
for key in ("HF_TOKEN",):
    value = os.environ.get(key)
    if not value and IN_COLAB:
        from google.colab import userdata
        try:
            value = userdata.get(key)
        except Exception:
            value = None
    if not value:
        value = getpass(f"{key}: ")
    if not value:
        raise RuntimeError(f"Missing {key}")
    os.environ[key] = value
del value
print("HF_TOKEN is present. Artifacts and metadata use HF only.")


In [ ]:
import pandas as pd
from huggingface_hub import HfApi
from gated_dual_ema_msd.cli.matrix import MatrixJob
from gated_dual_ema_msd.config.experiments import EXPERIMENTS, DATASET_MAX_LENGTHS, MODEL_NAME, MODEL_REVISION
from gated_dual_ema_msd.operations.notebook_batch import prepare_data, bind_manifest, run_batch, train_command, write_json
from gated_dual_ema_msd.training.r2_runtime import environment_metadata

if MODEL_NAME != PINNED_MODEL_NAME or MODEL_REVISION != PINNED_MODEL_REVISION:
    raise ValueError("Backbone differs from the frozen CafeBERT source")
if DATASET_MAX_LENGTHS["vinli"] != VINLI_MAX_LENGTH:
    raise ValueError("ViNLI max length must be 512")
for method in METHODS:
    experiment = EXPERIMENTS[method]
    if not experiment.use_ema or not experiment.use_msd or 42 not in experiment.seeds:
        raise ValueError(f"Architecture must preserve EMA/MSD/seed42: {method}")
JOBS = [MatrixJob(**spec) for spec in JOB_SPECS]
display(pd.DataFrame([dict(**spec, max_length=VINLI_MAX_LENGTH) for spec in JOB_SPECS]))
api = HfApi(token=os.environ["HF_TOKEN"])
HF_NAMESPACE = api.whoami()["name"]
CONFIG = {**HYPERPARAMS, "hf_namespace": HF_NAMESPACE, "hf_prefix": RUN_GROUP,
          "hf_private": HF_PRIVATE, "keep_local_checkpoints": False,
          "frozen_final": False, "test_peak_exploratory": True}
if previous and previous["config"] != CONFIG:
    raise ValueError("Settings differ from this RUN_GROUP. Choose a new RUN_GROUP.")

def verify_hf_write_access(jobs, config):
    for job in jobs:
        repo_id = (f"{config['hf_namespace']}/{config['hf_prefix']}-{job.dataset}-"
                   f"{job.experiment_id.lower().replace('_', '-')}-seed{job.seed}")
        api.create_repo(repo_id=repo_id, repo_type="model", private=config["hf_private"], exist_ok=True)
        if api.repo_info(repo_id=repo_id, repo_type="model").private != config["hf_private"]:
            raise RuntimeError(f"HF repository visibility differs: {repo_id}")
verify_hf_write_access(JOBS, CONFIG)
print("HF write access verified for", len(JOBS), "repositories; W&B disabled")


## Freeze the two-job recipe and pinned data
HF write access is checked for each run before training. The manifest binds
train/dev/test hashes, label order, code/model/data revisions and scheduler
settings. The helper's preparation flag includes test fingerprints; this
study keeps the existing exploratory test scans.


In [ ]:
from gated_dual_ema_msd.operations.architecture_search import environment_contract
DATA_FINGERPRINTS = prepare_data(REPO_DIR, DATASETS, frozen_final=True)
if DATA_FINGERPRINTS["vinli"]["revision"] != PINNED_VINLI_REVISION:
    raise ValueError("ViNLI differs from its pinned revision")
SESSION_ENVIRONMENT = environment_metadata()
ENVIRONMENT_CONTRACT = environment_contract(SESSION_ENVIRONMENT)
BATCH_MANIFEST = {"git_sha": SOURCE_SHA, "model_name": MODEL_NAME, "model_revision": MODEL_REVISION,
                  "precision": "bf16", "evaluation_precision": "fp32", "test_locked": False,
                  "environment_contract": ENVIRONMENT_CONTRACT, "label_order": ["E", "C", "N"],
                  "seeds": SEEDS, "datasets": DATASETS, "methods": METHODS,
                  "jobs": [dict(dataset=j.dataset, experiment_id=j.experiment_id, seed=j.seed) for j in JOBS],
                  "config": CONFIG, "execution": EXECUTION, "data_fingerprints": DATA_FINGERPRINTS,
                  "protocol": "vinli_relation_width_parallel_pilot",
                  "prior_test_exposure": True, "selection_policy": "exploratory_test_macro_f1",
                  "dev_role": "reference only; no dev gate or early stopping",
                  "test_peak_rule": {"weight_source": "ema", "metric": "macro_f1", "ties": "earliest_step"}}
SIGNATURE = bind_manifest(OUTPUT_ROOT, BATCH_MANIFEST)
write_json(OUTPUT_ROOT / "session_environment.json", SESSION_ENVIRONMENT)
print("Manifest:", SIGNATURE)
print("Split counts:", {s: x["row_count"] for s, x in DATA_FINGERPRINTS["vinli"]["splits"].items()})


## Preflight each head, then train both models concurrently
Preflights run one at a time and exit before training starts. Independent
processes own their seeds, CUDA allocator, optimizer, EMA, log and checkpoints.
Trainer logs stream below and remain in each Drive run folder as worker.log.
Publication runs one at a time; local weights are removed only after HF read-back.
The scheduler rechecks VRAM before launch and records its actual slot count.


In [ ]:
HEAD_PREFLIGHT = []
for job in JOBS:
    completed = subprocess.run(train_command(job, CONFIG, LOCAL_RUN_ROOT) + ["--preflight"],
                               cwd=REPO_DIR, text=True, capture_output=True, check=True)
    # The backbone prints loader status before the final pretty JSON report.
    report_start = completed.stdout.rfind("\n{")
    report = json.loads(completed.stdout[report_start + 1:] if report_start >= 0 else completed.stdout)
    experiment = report["experiment"]
    msd = report["msd"]
    if (report["train_precision"] != "bf16" or report["eval_precision"] != "fp32"
            or report["max_length"] != 512 or report["split_sizes"] != EXPECTED_SPLIT_SIZES
            or experiment["experiment_id"] != job.experiment_id
            or experiment["use_ema"] is not True or experiment["use_msd"] is not True
            or msd["msd_enabled"] is not True or msd["msd_num_paths"] != 5
            or msd["msd_shared_classifier"] is not True
            or msd["msd_dropout_probabilities"] != MSD_PROBABILITIES
            or report["parameters"]["additional_trainable_head_parameters"] <= 0):
        raise ValueError(f"Architecture preflight protocol mismatch: {job.experiment_id}")
    HEAD_PREFLIGHT.append(report)
write_json(OUTPUT_ROOT / "head_preflight.json", {"jobs": HEAD_PREFLIGHT})
display(pd.DataFrame([dict(experiment_id=r["experiment"]["experiment_id"],
                           head_parameters=r["parameters"]["additional_trainable_head_parameters"],
                           **r["split_sizes"]) for r in HEAD_PREFLIGHT]))


EXPECTED_HEAD_COUNTS = {"M3_FULL": 2761859, "ARCH_REL256": 3417347}
for report in HEAD_PREFLIGHT:
    method = report["experiment"]["experiment_id"]
    if report["parameters"]["additional_trainable_head_parameters"] != EXPECTED_HEAD_COUNTS[method]:
        raise ValueError(f"Unexpected relation-width head size: {method}")


In [ ]:
from gated_dual_ema_msd.operations.parallel_notebook_batch import run_parallel_batch
if BATCH_MANIFEST["execution"] != EXECUTION:
    raise ValueError("Execution settings changed after freeze. Use a new RUN_GROUP.")
run_parallel_batch(JOBS, REPO_DIR, LOCAL_RUN_ROOT, OUTPUT_ROOT, CONFIG,
                   BATCH_MANIFEST, **EXECUTION)


## Compare verified M3 and REL256 evidence
This reads saved predictions/curves and immutable HF references; it runs no
additional inference. Deltas appear only after both runs are verified.
Compare dev-selected test scores separately from exploratory test peaks.
Training times below include contention on the shared GPU; they are not
isolated architecture throughput benchmarks. No extra seeds launch here.


In [ ]:
from gated_dual_ema_msd.operations.relation_width_pilot import relation_width_table
COMPARISON = relation_width_table(OUTPUT_ROOT)
print(f"Verified runs: {len(COMPARISON)} / 2")
if not COMPARISON.empty:
    COMPARISON.to_csv(OUTPUT_ROOT / "m3_rel256_seed42_comparison.csv", index=False)
    display(COMPARISON)
if len(COMPARISON) != 2:
    print("Comparison pending: finish both verified seed-42 runs before choosing the next experiment")
print("Evidence, worker logs and resume directory:", OUTPUT_ROOT)
